In [1]:

from tnwater import load_gps, load_water_quality, merge_water_quality_with_gps, filter_ids


nut_df = load_water_quality("../data/wq_data_for_tennessee.csv")
gps = load_gps("../data/dam_distances.csv")
merged_df = merge_water_quality_with_gps(nut_df=nut_df, gps_df=gps)

# note that it converts any depths in feet to meters during the import



C:\Users\Spencer Womble\OneDrive\TN_Tech2\projects\reservoir_work\analysis_files\project_folder\tnwater\pipeline.py:30: DtypeWarning: Columns (0: End Date, 1: Measure Qualifier Code, 2: Measure Qualifier Description, 3: Quantitation Limit Unit Code, 4: Activity Depth Unit Code, 5: Analytical Method Identifier, 6: Result Speciation, 7: Result Value Type, 8: Result Detection Condition, 9: Result Status, 10: Quantitation Limit Type, 11: Quantitation Limit Speciation, 12: Sample Collection Method Identifier, 13: Sample Collection Method, 14: Sample Collection Method Description, 15: Sample Collection Method Context Code, 16: Sample Collection Method Context, 17: Sample Collection Equipment, 18: Start Time Zone Code, 19: End Time Zone Code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [2]:
# get ride of duplicate observations by random sampling (phosphorus seems to have a lot of duplicates)
group_keys = ['date_time', 'site', 'characteristic', 'sample_fraction']

deduped = (
    merged_df.sample(frac=1, random_state=42) # shuffle all rows - important to do first as we shuffle and then keep the first row for each duplicate
      .drop_duplicates(subset=group_keys, keep='first')
      .sort_values('date_time')
      .reset_index(drop=True)
)

In [3]:

# filter down to the location IDs that have good data coverage. The default contains the location IDs, but you can manually specify them if you want to.gps

# I'm retaining all here
filtered_df = filter_ids(df=deduped, ids=['cent_h1', 'cent_h2', 'cent_h3', 'cent_h4', 'cent_h5',
  'cent_h6', 'cent_h7', 'cent_h8', 'cent_h9', 'cent_h10',
  'cent_h11', 'cent_h12', 'cent_h13', 'cent_h14', 'cent_h15',
  'cent_h16', 'cent_h17', 'cent_h18', 'cent_h19', 'cent_h20',
  'dh1', 'dh2', 'dh3', 'dh4', 'dh5', 'dh6', 'dh7', 'dh8',
  'dh9', 'dh10', 'dh11', 'dh12', 'dh13', 'dh14', 'dh15',
  'dh16', 'dh17', 'dh18',
  'pp1', 'pp2', 'pp3', 'pp4', 'pp5', 'pp6', 'pp7', 'pp8',
  'pp9', 'pp10', 'pp11', 'pp12', 'pp13', 'pp14', 'pp15', 'pp16'])


In [105]:
# check if all nutrient/tss values are in mg/L
mask = (filtered_df["measure_unit_code"] == "mg/l") & (filtered_df["sample_fraction"].notna())
filtered_df[mask]["measure_unit_code"].unique()



<StringArray>
['mg/l']
Length: 1, dtype: str

In [10]:
# set NOx and TP values below or equal to the detection limit to half the detection limit
# note that this will apply to both total and dissolved phosphorus measurements given the 
# way the data is structured (there's a total vs dissolved column and TP is coded as "Phosphorus").
# We're not using dissolved P data, so it won't effect our analysis, but its not as clean as it could be.

import pandas as pd

MERGE_KEYS = ['monitoring_location_identifier', 'date_time', 'characteristic']

def load_mdl(path, characteristic, mdl_colname, extra_renames=None):
    rename_map = {
        'station': 'monitoring_location_identifier',
        'quantitation_limit_value': mdl_colname,
    }
    if extra_renames:
        rename_map.update(extra_renames)

    df = pd.read_csv(path).rename(columns=rename_map)
    df['characteristic'] = characteristic
    df['date_time'] = pd.to_datetime(
        df['date_time'].str.replace(', ', ' '),
        format='%Y%m%d %H%M',
    )

    df = (
        df.sample(frac=1, random_state=42)
          .drop_duplicates(subset=['date_time', 'monitoring_location_identifier'],
                           keep='first')
          .sort_values('date_time')
          .reset_index(drop=True)
    )
    return df[MERGE_KEYS + [mdl_colname]]


nox_mdl = load_mdl(
    '../data/mdl_with_gps.csv',
    characteristic='Inorganic nitrogen (nitrate and nitrite)',
    mdl_colname='true_nox_mdl',
    extra_renames={'datetime': 'date_time'},
)

tp_mdl = load_mdl(
    '../data/mdl_tp.csv',
    characteristic='Phosphorus',
    mdl_colname='true_tp_mdl',
)

filtered_df = (
    filtered_df
      .merge(nox_mdl, on=MERGE_KEYS, how='left')
      .merge(tp_mdl,  on=MERGE_KEYS, how='left')
)


# Preserve the raw lab value and create a half-DL substituted version
filtered_df['measure_value_half_dl'] = filtered_df['measure_value']

# Unified MDL column: each row has at most one MDL since merges are on characteristic
filtered_df['mdl'] = filtered_df['true_nox_mdl'].combine_first(filtered_df['true_tp_mdl'])

# Replace values at/below the detection limit with half the detection limit
below = filtered_df['measure_value'] <= filtered_df['mdl']
filtered_df.loc[below, 'measure_value_half_dl'] = filtered_df.loc[below, 'mdl'] / 2



# Replace values at/below the detection limit with half the detection limit
#for mdl_col in ('true_nox_mdl', 'true_tp_mdl'):
 #   below = filtered_df['measure_value'] <= filtered_df[mdl_col]
 #   filtered_df.loc[below, 'measure_value_half_dl'] = filtered_df.loc[below, mdl_col] / 2

In [14]:
# convert outliers to NA for NOx and TP

thresholds = {
    'Inorganic nitrogen (nitrate and nitrite)': 20,
    # add more pairs here as needed
    # 'Phosphorus': 5,
    # 'Kjeldahl nitrogen': 10,
}

threshold_col = filtered_df['characteristic'].map(thresholds)
above = filtered_df['measure_value'] > threshold_col

filtered_df = filtered_df.loc[~above].reset_index(drop=True)

In [108]:
filtered_df['activity_depth_value'].isna().sum()

np.int64(284)

In [15]:
# replace blanks/NAs for depth with 0 to indiciate that they are surface samples
# It's important to do this step before filtering to keep only the shallowest measurements for the depth profile samples
# as pandas will shift all NAs to the end, throwing the sorting order out of alignment

filtered_df['activity_depth_value'] = filtered_df['activity_depth_value'].fillna(0)

In [16]:
# Task 1: Keep all single-sample rows + only the shallowest row from depth profiles
filtered_surface_df = (
    filtered_df
    .sort_values(['id', 'date_time', 'activity_depth_value'])
    .drop_duplicates(subset=['id', 'date_time'], keep='first')
)




In [113]:
# save this long version to a csv
filtered_surface_df.to_csv('../data/cleaned_data_long.csv', index = False)


In [ ]:
# Task 2: Extract only rows from id/date_time combinations that have multiple depth measurements
depth_profiles_df = filtered_df[
    filtered_df
    .groupby(['id', 'date_time'])['activity_depth_value']
    .transform('size') > 1].copy()


depth_profiles_df.to_csv('../data/depth_profile_subset.csv', index = False)

In [ ]:
from tnwater import pivot_wider

# pivot data to wide format for plotting and modeling
filtered_df_wide = pivot_wider(filtered_surface_df)


In [36]:
# remove the "measure_value" prefix
filtered_df_wide.columns = filtered_df_wide.columns.str.replace("^measure_value_", "", regex=True)

In [37]:
from tnwater import clean_column_names
# clean column names for the new nutrient variable columns

filtered_df_wide = clean_column_names(filtered_df_wide)



In [38]:
# filter to retain only data collected no deeper than 1 meter

filtered_df_wide = filtered_df_wide[filtered_df_wide['activity_depth_value'] <= 1]



In [41]:
# check for outliers in no3_no2 column

from tnwater import outlier_check

outlier_check(filtered_df_wide, 
              sort_col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
              keep_col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
              rows = 5)

,inorganic_nitrogen_(nitrate_and_nitrite)_total
510,549.000
219,4.300
211,4.000
205,3.278
213,3.200


In [42]:
# check for outliers in tp column

from tnwater import outlier_check

outlier_check(filtered_df_wide, 
              sort_col='phosphorus_total',
              keep_col='phosphorus_total',
              rows = 5)

,phosphorus_total
3600,6.4000
24,0.7413
180,0.6500
205,0.6367
3264,0.6080


In [43]:

# check for outliers in chlorophyll_a_total column

from tnwater import outlier_check


outlier_check(filtered_df_wide, 
              sort_col='chlorophyll_a_total',
              keep_col='chlorophyll_a_total',
              rows = 5)

,chlorophyll_a_total
3090,28.20
3994,25.42
3533,24.30
3201,19.60
3203,16.20


In [44]:
# check for outliers in tss column

from tnwater import outlier_check


outlier_check(filtered_df_wide, 
              sort_col='total_suspended_solids_suspended',
              keep_col='total_suspended_solids_suspended',
              rows = 5)

,total_suspended_solids_suspended
250,79.0
1804,61.6
225,54.0
2565,25.3
216,25.0


In [45]:
from tnwater import outlier_to_na

# outlier thresholds based on professional judgement and data exploration
# there were no obvious outliers for chla or tss - at least on the high end

# convert the outliers to NA for no3_no2
filtered_df_wide=outlier_to_na(filtered_df_wide, 
                          col='inorganic_nitrogen_(nitrate_and_nitrite)_total',
                          greater_than_threshold=20)

# convert the outliers to NA for TP
filtered_df_wide=outlier_to_na(filtered_df_wide, 
                          col='phosphorus_total',
                          greater_than_threshold=3)

In [46]:
# write filtered wide data frame to new csv

filtered_df_wide.to_csv('../data/cleaned_data.csv', index=False)